# Tugas Besar 2 IF3170 Inteligensi Artifisial

*Klasifikasi pembatalan pemesanan hotel (`is_canceled`).*

## IF3170_G08_Teazzi

- 13524034 - Jason Edward Salim
- 13524064 - Stefani Angeline Oroh
- 13524067 - Bryan Pratama Putra Hendra
- 13524079 - Angelina Andra Alanna
- 13524106 - Michael James Liman

## Table of Contents (CRISP-DM)

1. [**Business Understanding**](#1)
2. [**Data Understanding**](#2)
3. [**Data Preparation**](#3)
4. [**Modeling**](#4)
5. [**Evaluation**](#5)
6. [**Deployment**](#6)

## Initialization

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "src" else Path.cwd() / "src"))

from config import Settings, seed_everything
from eda import types, missing, duplicates, pie, bar, histogram, boxplot, iqr_outliers, target_rate, correlation
from preprocessing import load_data, split, drop_duplicates, Preprocessor, Pipeline
from validation import fit, holdout, kfold, compare, tune, best, importance, confusion, report, misclassified, submit, with_threshold, never_cancel
from models import KNN, LogisticRegression, SVM, DTL, BayesNetwork

seed_everything()

In [ ]:
train, test = load_data()
pd.Series({"train": len(train), "test": len(test)})

---

# Business Understanding <a name="1"></a>

---

Memprediksi apakah sebuah pemesanan hotel akan dibatalkan (`is_canceled = 1`) atau tidak (`is_canceled = 0`). Metrik evaluasi: macro F1-score.

> TODO: konteks bisnis dan dampak false positive / false negative.

---

# Data Understanding <a name="2"></a>

---

## Data Overview

In [ ]:
train

In [ ]:
display(types(train))
display(types(test))

In [ ]:
missing(train, test)

In [ ]:
duplicates(train, test)

## Target Distribution

In [ ]:
pie(train)
bar(train)

## Feature Analysis

In [ ]:
num_col = train.select_dtypes("number").columns.drop(Settings.TARGET)
cat_col = train.columns.difference(num_col).drop([Settings.ID, Settings.TARGET])

display(train[num_col].describe().T)
display(train[cat_col].describe().T)

In [ ]:
histogram(train, num_col)
boxplot(train, num_col)

In [ ]:
iqr_outliers(train, num_col)

In [ ]:
target_rate(train, list(cat_col) + ['special_requests', 'arrival_month', 'is_returning_guest'])

In [ ]:
correlation(train)

---

# Data Preparation <a name="3"></a>

---

Data dibagi sebelum preprocessing. Seluruh langkah di-fit hanya pada `X_train`, lalu diterapkan ke `X_train`, `X_val`, dan `X_test`. Logika ada di `src/preprocessing.py` (fitur baru di `engineer`).

Tiap model memakai preprocessing yang paling cocok untuknya, dipilih lewat macro F1 5-fold CV: `default` untuk BN, `svm` untuk SVM, `tree` (tanpa scaling dan capping) untuk DTL, `knn` (dengan fitur baru dan penanda missing `special_requests`) untuk KNN, dan `lr` (one-hot dari bin kuantil) untuk LR. Preprocessing `svm` menambah dua kelompok fitur berbasis label. Pertama, `near_count`/`near_rate` (dan `near2_*` dengan toleransi harga lebih longgar): banyaknya baris latih yang tampak seperti pesanan grup yang sama (tanggal kedatangan, lama menginap, channel, kamar, dan lainnya sama; lead time selisih ≤ 10 hari; harga selisih ≤ 3% atau ≤ 10%) beserta tingkat pembatalannya. Pasangan seperti ini sama labelnya pada 95% kasus. Kedua, target encoding `te_*`: tingkat pembatalan baris latih dengan kunci yang sama, misalnya bulan kedatangan × pita lead time. Untuk baris latih, keduanya dihitung out-of-fold sehingga sebuah baris tidak pernah melihat labelnya sendiri. Kunci dengan pita harga (`price10`, harga per 10%) ikut dipakai. Macro F1 5-fold SVM naik dari 0,838 menjadi 0,868.

`X_trains`, `X_vals`, dan `X_tests` memetakan nama model ke data hasil preprocessing-nya.

In [ ]:
X_train, X_val, y_train, y_val = split(train)
X_test = test.drop(columns=[Settings.ID])

X_train, y_train = drop_duplicates(X_train, y_train)
pd.Series({"X_train": len(X_train), "X_val": len(X_val), "X_test": len(X_test)})

In [ ]:
CAP = ["days_before_arrival", "nightly_rate_idr"]
ORDINAL = {
    "meal_package": ["Room Only", "Breakfast", "Half Board", "Full Board"],
    "room_category": [f"Type {c}" for c in "ABCDEFG"],
}
# Features from preprocessing.py that only helped KNN in 5-fold CV, dropped for the other models
NEW_FEATURES = ["arrival_weekday", "booking_month", "booking_weekday", "price_per_guest", "nightly_rate_idr_ratio", "days_before_arrival_ratio"]
# Target encodings for the SVM: lead_band = lead time bands, lead20 = 20-day lead bins, price10 = 10%-wide price bins,
# nights = weekend + weekday nights
TARGET_ENCODE = [
    ("arrival_year", "arrival_month", "lead_band"), ("nights", "booking_channel", "lead_band"), ("date", "lead_band"),
    ("week", "lead_band"), ("date", "booking_channel"), ("room_category", "lead_band"), ("special_requests", "lead_band"),
    ("booking_channel", "room_category"), ("arrival_year", "arrival_month", "booking_channel"), ("date", "nights"),
    ("meal_package", "lead_band"), ("num_adults", "booking_channel", "lead_band"), ("lead20", "special_requests", "booking_channel"),
    ("lead20", "arrival_year", "arrival_month"), ("lead_band", "special_requests", "booking_channel", "arrival_year"),
    ("lead20", "room_category", "special_requests"), ("price10", "lead_band"), ("price10", "room_category", "arrival_year"),
    ("price10", "special_requests", "booking_channel"), ("price10", "arrival_year", "arrival_month"),
]
BINNED = ["days_before_arrival", "nightly_rate_idr", "arrival_month", "date_rate", "special_requests", "total_nights",
          "arrival_weekday", "booking_month", "price_per_guest", "num_adults"]

preps = {
    "default": Preprocessor(
        zero_as_missing=[],                                   # e.g. ["nightly_rate_idr"]
        num_strategy="median",
        cap=CAP,
        ordinal=ORDINAL,
        scaler="standard",                                    # 'standard' | 'minmax' | 'robust' | None
        drop=NEW_FEATURES,
    ),
    # + group-booking neighbours (near_*) and out-of-fold target encodings (te_*)
    "svm": Preprocessor(cap=CAP, ordinal=ORDINAL, flag_missing=["special_requests"], near_dup=True,
                        near_tols=[(0, 10, 0.03), (0, 10, 0.10)], target_encode=TARGET_ENCODE, te_smoothing=5),
    "tree": Preprocessor(ordinal=ORDINAL, scaler=None, drop=NEW_FEATURES),                     # tree splits ignore scale
    "knn": Preprocessor(cap=CAP, ordinal=ORDINAL, flag_missing=["special_requests"]),
    "lr": Preprocessor(cap=CAP, flag_missing=["special_requests"], binned=BINNED, n_bins=20),   # bins let a linear model bend
}
uses = {"svm": "svm", "bn": "default", "bn_kdb": "default", "dtl": "tree", "knn": "knn", "lr": "lr",
        "sk_svm": "svm", "sk_nb": "default", "sk_dtl": "tree", "sk_knn": "knn", "sk_lr": "lr"}

prepared = {name: (p.fit_transform(X_train, y_train), p.transform(X_val), p.transform(X_test)) for name, p in preps.items()}
X_trains, X_vals, X_tests = ({model: prepared[prep][i] for model, prep in uses.items()} for i in range(3))
X_train, X_val, X_test = prepared["default"]
X_train.head()

#### Keputusan

> TODO: justifikasi imputasi, capping, encoding, scaling.

---

# Modeling <a name="4"></a>

---

Model from scratch ada di `src/models/` (implementasi `fit`/`predict`). Model sklearn dipakai sebagai pembanding. `METHOD` menentukan penanganan class imbalance (`None`, `'smote'`, `'undersample'`), hanya diterapkan pada data latih.

In [ ]:
METHOD = None

# Pipeline(model, select=k, pca=n, ranker=m) adds per-model feature selection (top k by m's feature importance) and PCA
ranker = DTL(max_depth=10, min_samples_split=50)

# Settings chosen by 5-fold CV macro F1 on train.csv
models = {
    "knn": Pipeline(KNN(k=5, metric="manhattan"), select=12, ranker=ranker),
    "lr": LogisticRegression(lr=0.5, n_iter=3000),
    "svm": SVM(kernel="rbf", C=2, gamma=0.025),
    "dtl": DTL(criterion="entropy", max_depth=10),
    "bn": BayesNetwork(),
    "bn_kdb": BayesNetwork(n_bins=5, k=10),   # KDB: maksimal k parent fitur
}

In [ ]:
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

sk_models = {
    "sk_knn": Pipeline(KNeighborsClassifier(5, metric="manhattan"), select=12, ranker=ranker),
    "sk_lr": SkLogisticRegression(max_iter=3000),
    "sk_svm": SVC(kernel="rbf", C=2, gamma=0.025),
    "sk_dtl": DecisionTreeClassifier(criterion="entropy", max_depth=10, random_state=Settings.SEED),
    "sk_nb": GaussianNB(),
}

## SVM Feature Selection

SVM diselesaikan dalam bentuk dual dengan SMO, sehingga untuk kernel linear w = Σ αᵢyᵢxᵢ. Fitur diranking berdasarkan |w| SVM linear yang dilatih pada seluruh fitur, lalu `select` fitur teratas dipakai untuk melatih ulang SVM. Model final memakai kernel RBF, yang tidak memiliki w eksplisit.

In [ ]:
svm_fitted = fit(Pipeline(SVM(kernel="linear", C=0.1), select=10), X_train, y_train)
pd.Series(X_train.columns[svm_fitted.keep], name="svm selected features")

---

# Evaluation <a name="5"></a>

---

## Model Comparison

Holdout pada `X_val` dan K-Fold pada `X_train` dengan macro F1-score.

In [ ]:
all_models = {**models, **sk_models}
compare(holdout(all_models, X_trains, y_train, X_vals, y_val, METHOD), kfold(all_models, X_trains, y_train, METHOD))

## Hyperparameter Tuning

In [ ]:
grids = {
    "knn": {"k": [5, 7, 11], "metric": ["euclidean", "manhattan"]},
    "lr": {"lr": [0.1, 0.5], "n_iter": [1000, 3000]},
    "svm": {"kernel": ["rbf"], "C": [1, 2], "gamma": [0.025, 0.035]},
    "dtl": {"max_depth": [8, 10, 12], "min_samples_split": [2, 20]},
    "bn": {"n_bins": [3, 5, 10]},
    "bn_kdb": {"k": [6, 8, 10]},
}

grid_tables = {name: tune(models[name], grid, X_trains[name], y_train, METHOD) for name, grid in grids.items()}
tuned_models = {name: best(models[name], table) for name, table in grid_tables.items()}
grid_tables["knn"].head()

In [ ]:
ranking = compare(
    holdout(tuned_models, X_trains, y_train, X_vals, y_val, METHOD),
    kfold(tuned_models, X_trains, y_train, METHOD),
)
ranking

## Error Analysis

In [ ]:
# Pakai model from scratch terbaik, selama belum ada yang diimplementasikan, pakai baseline sklearn
if ranking.empty:
    ranking = compare(holdout(sk_models, X_trains, y_train, X_vals, y_val, METHOD), kfold(sk_models, X_trains, y_train, METHOD))
    tuned_models = {**tuned_models, **sk_models}

best_name = ranking.index[0]
best_model = tuned_models[best_name]
best_name, best_model

In [ ]:
confusion(tuned_models, X_trains, y_train, X_vals, y_val, METHOD)

In [ ]:
report(tuned_models, X_trains, y_train, X_vals, y_val, METHOD)

In [ ]:
wrong_rows = misclassified(best_model, train, X_trains[best_name], y_train, X_vals[best_name], y_val, METHOD)
print(f"{len(wrong_rows)} dari {len(X_val)} baris salah")
wrong_rows["predicted"].map({1: "false positive", 0: "false negative"}).value_counts()

In [ ]:
importance(sk_models["sk_dtl"], X_trains["sk_dtl"], y_train).head(10)

In [ ]:
from models.dtl import plot_tree

# top_n = number of split levels from the root drawn in the image
dtl_fitted = fit(tuned_models["dtl"], X_trains["dtl"], y_train, METHOD)
fig = plot_tree(dtl_fitted, list(X_trains["dtl"].columns), top_n=3, path=Settings.OUTPUT_DIR / "dtl_tree.png")
fig

In [ ]:
# Bandingin tiga struktur Bayesian Network: Naive Bayes, TAN (1 parent fitur), KDB (sampai k parent)
bn_variants = {
    "bn_nb": BayesNetwork(n_bins=5),
    "bn_tan": BayesNetwork(n_bins=5, tree=True),
    "bn_kdb": BayesNetwork(n_bins=5, k=10),
}
compare(holdout(bn_variants, X_train, y_train, X_val, y_val, METHOD), kfold(bn_variants, X_train, y_train, METHOD))

In [ ]:
from models.bayes_graph import plot_bayes_graph

# Graf pakai struktur TAN karena tiap fitur cuma punya satu parent fitur
bn_tan = fit(BayesNetwork(n_bins=5, tree=True), X_train, y_train, METHOD)
fig = plot_bayes_graph(bn_tan, list(X_train.columns), class_name=Settings.TARGET)
fig.savefig(Settings.OUTPUT_DIR / "bayes_graph.png", dpi=160, bbox_inches="tight")
fig

In [ ]:
from validation import fit_predict, macro_f1

cost_variants = {
    "nb": BayesNetwork(n_bins=5),
    "nb + biaya": BayesNetwork(n_bins=5, cost_fp=2),
    "tan": BayesNetwork(n_bins=5, tree=True),
    "tan + biaya": BayesNetwork(n_bins=5, tree=True, cost_fp=2),
    "kdb": BayesNetwork(n_bins=5, k=10),
    "kdb + biaya": BayesNetwork(n_bins=5, k=10, cost_fp=2),
}

rows = {}
for name, model in cost_variants.items():
    # latih di train, prediksi di val
    predicted, actual = fit_predict(model, X_train, y_train, X_val, METHOD), y_val.values
    fp, fn = ((predicted == 1) & (actual == 0)).sum(), ((predicted == 0) & (actual == 1)).sum()
    rows[name] = {"FP": fp, "FN": fn, "biaya (2 FP + FN)": 2 * fp + fn, "macro F1": macro_f1(actual, predicted)}
pd.DataFrame(rows).T.astype({"FP": int, "FN": int, "biaya (2 FP + FN)": int})

## Bonus Optimisasi Logistic Regression

Model `lr` pada bagian Modeling tetap memakai implementasi from-scratch di `src/models/logistic_regression.py`. Eksperimen ini membandingkan batch GD, mini-batch SGD, dan L-BFGS from scratch dengan fitur c010 yang sama. Evaluasi memakai stratified holdout 20% (seed 42), regularisasi L2 0.1, dan threshold 0.42. Cell berikut menampilkan macro F1, accuracy, log-loss, dan status solver, serta menghasilkan tiga CSV prediksi dan `lr_c010_solver_comparison.csv`.

> Referensi L-BFGS: Liu dan Nocedal (1989), *Mathematical Programming* 45, 503-528, [PDF mirror Florida State University](https://people.sc.fsu.edu/~inavon/5420a/liu89limited.pdf). Saratchandran et al. (2023), arXiv:2305.08552, section 3.2, [HTML](https://arxiv.org/html/2305.08552).
> Line search Armijo dan Strong Wolfe: [Wolfram Language Documentation](https://reference.wolframcloud.com/language/tutorial/UnconstrainedOptimizationStepControl.html). Rujukan asli Armijo (1966), *Pacific Journal of Mathematics* 16(1), 1-3, [PDF langsung](https://msp.org/pjm/1966/16-1/pjm-v16-n1-p01-p.pdf).
> Gradient descent dan variannya: [GeeksforGeeks](https://www.geeksforgeeks.org/machine-learning/gradient-descent-algorithm-and-its-variants/) dan [IBM Think](https://www.ibm.com/think/topics/gradient-descent).

In [ ]:
from logistic_submission import run_three_solver_submissions

lr_solver_results, lr_solver_outputs = run_three_solver_submissions()
lr_solver_results[["model", "validation_macro_f1", "validation_accuracy", "validation_log_loss", "iterations", "converged"]]

---

# Deployment <a name="6"></a>

---

Model terbaik dilatih ulang dengan seluruh `train.csv`, dan preprocessing-nya juga di-fit ulang pada seluruh data latih. Ambang skor (threshold) dipilih dari skor out-of-fold 5-fold agar macro F1 maksimal. Di tiap fold, preprocessing di-fit ulang hanya pada data latih fold tersebut, sehingga fitur berbasis label (`date_rate`, `te_*`, `near_*`) tidak membocorkan label baris yang sedang dinilai. Baris dengan `special_requests` ≥ 3 atau `booking_channel` Complimentary diprediksi 0 karena tidak ada satu pun yang batal pada data latih. Prediksi `X_test` disimpan ke `output/<model>_<timestamp>.csv` beserta `.pkl` modelnya.


In [ ]:
raw_full, y_full = train.drop(columns=[Settings.ID, Settings.TARGET]), train[Settings.TARGET]
final_prep = preps[uses[best_name]]

final_model = with_threshold(best_model, raw_full, y_full, METHOD, prep=final_prep)
print(f"threshold: {final_model.threshold:.4f}")

X_full = final_prep.fit_transform(raw_full, y_full)
X_test_full = final_prep.transform(test.drop(columns=[Settings.ID]))
submit(final_model, best_name, X_full, y_full, X_test_full, test[Settings.ID], METHOD, force_zero=never_cancel(test))


In [ ]:
kdb_prep = preps[uses["bn_kdb"]]
X_full_kdb = kdb_prep.fit_transform(raw_full, y_full)
X_test_kdb = kdb_prep.transform(test.drop(columns=[Settings.ID]))
submit(BayesNetwork(n_bins=5, k=10), "bn_kdb10", X_full_kdb, y_full, X_test_kdb, test[Settings.ID], METHOD)